In [13]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType
from pyspark.sql.functions import window, sum as spark_sum, count, col
from datetime import datetime, timedelta
import os, shutil, time
import pandas as pd

spark = SparkSession.builder \
    .appName("Tugas10-KopiNusantara") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")
print("SparkSession siap. Versi Spark:", spark.version)

SparkSession siap. Versi Spark: 3.5.6


In [14]:
# Bersihkan sisa percobaan sebelumnya agar hasil selalu bersih
folder_stream = "tugas10_stream"
for f in [folder_stream, "tugas10_output", "tugas10_checkpoint"]:
    shutil.rmtree(f, ignore_errors=True)
os.makedirs(folder_stream, exist_ok=True)

skema_pesanan = StructType([
    StructField("pesanan_id", StringType(), True),
    StructField("nama_cabang", StringType(), True),
    StructField("nama_menu", StringType(), True),
    StructField("qty", IntegerType(), True),
    StructField("harga_satuan", IntegerType(), True),
    StructField("waktu_pesan", TimestampType(), True),
])

df_stream = spark.readStream \
    .format("csv") \
    .schema(skema_pesanan) \
    .option("header", True) \
    .load(folder_stream)

print("isStreaming:", df_stream.isStreaming)
df_stream.printSchema()

isStreaming: True
root
 |-- pesanan_id: string (nullable = true)
 |-- nama_cabang: string (nullable = true)
 |-- nama_menu: string (nullable = true)
 |-- qty: integer (nullable = true)
 |-- harga_satuan: integer (nullable = true)
 |-- waktu_pesan: timestamp (nullable = true)



In [15]:
df_total = df_stream.withColumn("total_harga", col("qty") * col("harga_satuan"))

agregasi_cabang = df_total.groupBy("nama_cabang").agg(
    spark_sum("total_harga").alias("total_pendapatan"),
    count("pesanan_id").alias("jumlah_pesanan")
)

query1 = agregasi_cabang.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("agregasi_cabang") \
    .start()

print("Query agregasi_cabang aktif:", query1.isActive)

Query agregasi_cabang aktif: True


In [16]:
df_stream_w = spark.readStream \
    .format("csv") \
    .schema(skema_pesanan) \
    .option("header", True) \
    .load(folder_stream)

agregasi_waktu = df_stream_w \
    .withColumn("total_harga", col("qty") * col("harga_satuan")) \
    .groupBy(window(col("waktu_pesan"), "2 minutes")) \
    .agg(spark_sum("total_harga").alias("total_pendapatan"))

query2 = agregasi_waktu.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("agregasi_waktu") \
    .start()

print("Query agregasi_waktu aktif:", query2.isActive)

Query agregasi_waktu aktif: True


In [17]:
def kirim_batch(nomor, baris):
    """Tulis satu file CSV baru ke folder streaming."""
    pd.DataFrame(baris, columns=[
        "pesanan_id", "nama_cabang", "nama_menu", "qty", "harga_satuan", "waktu_pesan"
    ]).to_csv(f"{folder_stream}/batch_{nomor}.csv", index=False)

def tampil_cabang():
    spark.sql("SELECT * FROM agregasi_cabang ORDER BY nama_cabang").show()

fmt = lambda dt: dt.strftime("%Y-%m-%d %H:%M:%S")
t0 = datetime.now()

# ---- Batch 1: Magelang & Yogyakarta
kirim_batch(1, [
    ["P001", "Magelang",   "Kopi Susu",  2, 18000, fmt(t0)],
    ["P002", "Yogyakarta", "Americano",  1, 20000, fmt(t0)],
    ["P003", "Magelang",   "Latte",      1, 25000, fmt(t0)],
])
print("Batch 1 dikirim."); time.sleep(8)
tampil_cabang()

Batch 1 dikirim.


+-----------+----------------+--------------+
|nama_cabang|total_pendapatan|jumlah_pesanan|
+-----------+----------------+--------------+
|   Magelang|           61000|             2|
| Yogyakarta|           20000|             1|
+-----------+----------------+--------------+



In [6]:
# ---- Batch 2: Magelang bertambah, Solo muncul
kirim_batch(2, [
    ["P004", "Magelang", "Cappuccino", 3, 22000, fmt(t0 + timedelta(seconds=30))],
    ["P005", "Solo",     "Kopi Susu",  4, 18000, fmt(t0 + timedelta(seconds=30))],
])
print("Batch 2 dikirim."); time.sleep(8)
tampil_cabang()

Batch 2 dikirim.


+-----------+----------------+--------------+
|nama_cabang|total_pendapatan|jumlah_pesanan|
+-----------+----------------+--------------+
|   Magelang|          127000|             3|
|       Solo|           72000|             1|
| Yogyakarta|           20000|             1|
+-----------+----------------+--------------+



In [7]:
# ---- Batch 3: Yogyakarta & Solo bertambah (waktu +3 menit -> window berbeda)
kirim_batch(3, [
    ["P006", "Yogyakarta", "Latte",     2, 25000, fmt(t0 + timedelta(minutes=3))],
    ["P007", "Solo",       "Americano", 1, 20000, fmt(t0 + timedelta(minutes=3))],
    ["P008", "Yogyakarta", "Kopi Susu", 5, 18000, fmt(t0 + timedelta(minutes=3))],
])
print("Batch 3 dikirim."); time.sleep(8)
tampil_cabang()

print("Hasil agregasi window 2 menit:")
spark.sql("""SELECT window.start, window.end, total_pendapatan
             FROM agregasi_waktu ORDER BY window.start""").show(truncate=False)

Batch 3 dikirim.


+-----------+----------------+--------------+
|nama_cabang|total_pendapatan|jumlah_pesanan|
+-----------+----------------+--------------+
|   Magelang|          127000|             3|
|       Solo|           92000|             2|
| Yogyakarta|          160000|             3|
+-----------+----------------+--------------+

Hasil agregasi window 2 menit:
+-------------------+-------------------+----------------+
|start              |end                |total_pendapatan|
+-------------------+-------------------+----------------+
|2026-11-11 10:46:00|2026-11-11 10:48:00|219000          |
|2026-11-11 10:48:00|2026-11-11 10:50:00|160000          |
+-------------------+-------------------+----------------+



In [18]:
query3 = df_stream.writeStream \
    .format("parquet") \
    .option("path", "tugas10_output") \
    .option("checkpointLocation", "tugas10_checkpoint") \
    .outputMode("append") \
    .start()

# Batch baru (batch 4)
kirim_batch(4, [
    ["P009", "Magelang", "Americano", 2, 20000, fmt(datetime.now())],
    ["P010", "Solo",     "Latte",     1, 25000, fmt(datetime.now())],
])
print("Batch 4 dikirim."); time.sleep(10)

print("Isi folder tugas10_output:")
!ls tugas10_output

Batch 4 dikirim.


Isi folder tugas10_output:
part-00000-b01d2854-4bcb-4bb9-b2d2-0b2e981f5495-c000.snappy.parquet
part-00001-e99a6210-1f14-445f-8da2-447902a6984b-c000.snappy.parquet
_spark_metadata


In [10]:
# Baca kembali sebagai DataFrame batch biasaquery1.stop()
query2.stop()
query3.stop()
print("Status aktif:", query1.isActive, query2.isActive, query3.isActive)
spark.stop()
print("SparkSession ditutup.")
df_hasil = spark.read.parquet("tugas10_output")
print("Jumlah baris:", df_hasil.count())
df_hasil.orderBy("pesanan_id").show(truncate=False)

Jumlah baris: 10
+----------+-----------+----------+---+------------+-------------------+
|pesanan_id|nama_cabang|nama_menu |qty|harga_satuan|waktu_pesan        |
+----------+-----------+----------+---+------------+-------------------+
|P001      |Magelang   |Kopi Susu |2  |18000       |2026-11-11 10:46:46|
|P002      |Yogyakarta |Americano |1  |20000       |2026-11-11 10:46:46|
|P003      |Magelang   |Latte     |1  |25000       |2026-11-11 10:46:46|
|P004      |Magelang   |Cappuccino|3  |22000       |2026-11-11 10:47:16|
|P005      |Solo       |Kopi Susu |4  |18000       |2026-11-11 10:47:16|
|P006      |Yogyakarta |Latte     |2  |25000       |2026-11-11 10:49:46|
|P007      |Solo       |Americano |1  |20000       |2026-11-11 10:49:46|
|P008      |Yogyakarta |Kopi Susu |5  |18000       |2026-11-11 10:49:46|
|P009      |Magelang   |Americano |2  |20000       |2026-11-11 10:48:49|
|P010      |Solo       |Latte     |1  |25000       |2026-11-11 10:48:49|
+----------+-----------+----------

In [11]:
query1.stop()
query2.stop()
query3.stop()
print("Status aktif:", query1.isActive, query2.isActive, query3.isActive)
spark.stop()
print("SparkSession ditutup.")

Status aktif: False False False
SparkSession ditutup.
